# E2 · The same columns, in another order: the model does not notice

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

See what happens when a model trained on numpy arrays receives in production a DataFrame with the same columns in a different order: **no error** and almost random predictions. Build a pipeline that selects the columns by name and
a schema checking function.

## Dataset

`load_wine(as_frame=True)`: 13 numeric columns, 3 classes, stratified 70/30 split: `X_tr, X_te, y_tr, y_te` (DataFrames). `X_prod` is the test set with the columns shuffled, as it would arrive from another system.

## Educational purpose

Recognize the "position instead of name" bug: with `.to_numpy()` or `.values` the information on the names disappears and the model reads the `k`-th column whatever it is. It is an integration problem, not a model problem, and it is very common.

## Task

1. Run the colleague's cell: `test_acc` is high, `bug_prod_acc` (same examples, shuffled columns) collapses, and no code reported an error.
2. Save `changed_positions`: how many of the 13 columns of `X_prod` are in a different position from `X_tr`.
3. Write `check_schema(X, expected)`: it raises `ValueError` if an expected column is missing or there is an extra one; otherwise it returns `X[list(expected)]` (columns reordered as in training).
4. Rebuild the model with a `ColumnTransformer` that **selects the columns by name** (`StandardScaler` on all the columns, given by name) and train it on the DataFrames: save `ok_model`, `ok_test_acc` and `ok_prod_acc` (on the DataFrame with the shuffled columns, without reordering them by hand).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

wine = load_wine(as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(wine.data, wine.target, test_size=0.3, stratify=wine.target, random_state=0)
columns = list(X_tr.columns)
prod_order = list(np.random.default_rng(1).permutation(columns))
X_prod = X_te[prod_order]                                    # same columns and same examples, different order

## Your work

### The colleague's code

In [ ]:
bug_model = make_pipeline(StandardScaler(), KNeighborsClassifier(7)).fit(X_tr.to_numpy(), y_tr)
test_acc = bug_model.score(X_te.to_numpy(), y_te)
bug_prod_acc = bug_model.score(X_prod.to_numpy(), y_te)
print(f"test accuracy: {test_acc:.3f} | in production (columns in a different order): {bug_prod_acc:.3f}")

### Your diagnosis and the fix

In [ ]:
changed_positions = None

In [ ]:
def check_schema(X, expected):
    ...

In [ ]:
ok_model = None
ok_test_acc = None
ok_prod_acc = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`ok_model = make_pipeline(ColumnTransformer([("num", StandardScaler(), columns)]), KNeighborsClassifier(7)).fit(X_tr, y_tr)`; `ok_model.score(X_prod, y_te)`. The `ColumnTransformer` selects by **name** and always produces the same output order. `check_schema`: `if set(X.columns) != set(expected): raise ValueError(...)`.'):
    check(test_acc > 0.9 and bug_prod_acc < 0.7, "the colleague's model: over 0.9 on the test set, below 0.7 with the shuffled columns, with no errors", "don't modify the colleague's code")
    check(changed_positions == sum(a != b for a, b in zip(columns, prod_order)) and changed_positions >= 10, "at least 10 columns out of 13 are in a different position", "compare the two lists position by position")
    check(list(check_schema(X_prod, columns).columns) == columns and (check_schema(X_prod, columns).to_numpy() == X_te.to_numpy()).all(), "check_schema reorders the columns as in training", "X[list(expected)]")
    for bad, name in [(X_prod.drop(columns=columns[0]), "missing column"), (X_prod.assign(extra=0.0), "extra column")]:
        try:
            check_schema(bad, columns)
            raised = False
        except ValueError:
            raised = True
        check(raised, f"check_schema raises ValueError with a {name}", "compare set(X.columns) with set(expected)")
    check(ok_model is not None and ok_prod_acc > 0.9 and abs(ok_prod_acc - ok_test_acc) < 1e-12, "with the selection by name the shuffled columns give the same result as the test set", "ColumnTransformer([('num', StandardScaler(), columns)])")
done()

---
The commented solution is in `solutions/E_find_the_bug/E2_columns_in_a_different_order_in_production.ipynb`. Open it after you pass the check, or after you have really tried.